# Chapter 15: Greedy and Sampling

[Read this chapter online](https://jackluu.io/book/section-5-generation/ch15-greedy-and-sampling/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch15-greedy-and-sampling.ipynb)

From *Teach Your Computer to Write: Build and Train an LLM from Zero* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 15: Greedy and Sampling

![The map with New Text highlighted](https://jackluu.io/book/assets/diagrams/ch15-where-we-are.png)

*Figure 15.1: We are ready to generate new text with the trained model.*

So far we've trained and loaded our model. Now we'd like to actually use it to generate some text. However, our model outputs a list of 65 scores (also known as logits), one for each possible character. We need to devise a strategy to choose a winner. In this chapter you will:

- Generate text by always choosing the highest-scoring character (greedy decoding).
- Generate text by choosing characters randomly based on their probabilities (sampling).
- Compare the trade-offs between deterministic and creative text generation.

> **Words to Know**
>
> - **Greedy Decoding**: always picking the single highest-probability token.
> - **Sampling**: choosing the next token randomly, giving higher-probability tokens a better chance to be selected.

## Theory

### Greedy Decoding

One method we can use is simply selecting the most likely token each time. This is known as greedy decoding.

Although we may want to select the "best" answer every time, the problem with this is we will frequently get into a loop. Suppose our model knows a phrase. It will select the next most likely token for that phrase. Now that token is included in the context so it will select the next token in that phrase. And so on it will repeat that phrase infinitely.

Since we are always choosing the most likely token greedy decoding is a deterministic method. Given the same prompt, we will receive the same output.

### Sampling

We can avoid getting stuck in loops with something called sampling. Rather than automatically choosing the highest-scoring character, we convert the raw scores (also called logits) to percentages (probabilities) and pick a winner at random.

![Argmax taking the tallest bar, and sampling from the distribution](https://jackluu.io/book/assets/diagrams/ch15-greedy-and-sampling.png)

*Figure 15.2: Greedy decoding always takes the tallest bar, and sampling draws randomly from the distribution.*

When there's a 60% chance of the letter e occurring it will occur 60% of the time. When x has a 1% chance it will be selected less often. But there's still a chance it can happen. Which is good as it breaks the loop and makes your text more interesting. Downside being you won't get the same text each time it runs because it will select a different draw. All code snippets in this book use a seed (`torch.manual_seed(42)`) to lock the draw so when you run it you'll receive the same text.

Generation takes New Text back to the Tokens step on our "where we are" map. We generate one character and append it to our prompt then loop back.

> **In Business**
>
> It will depend on what you want it to do with our house-style assistant. If you want it to tell you the answer to a question about your company's information, you would want it to use greedy decoding, because you want it to be as predictable as possible (greedy can still give you the wrong answer). If you want it to write a marketing email that represents your company, you would want sampling, so it can be more varied and come up with new ideas.

## Code

We implement both strategies in the same generation loop. The core difference is how `next_id` is chosen.

```python
def generate_greedy(model, prompt, encode, decode, cfg, max_new_tokens=200):
    # Always picks the token with the highest predicted score
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    for _ in range(max_new_tokens):
        ctx     = ids[:, -cfg.block_size:]
        logits  = model(ctx)
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids     = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

def generate_sample(model, prompt, encode, decode, cfg, max_new_tokens=200):
    # Picks the next token randomly based on the model's probabilities
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    for _ in range(max_new_tokens):
        ctx    = ids[:, -cfg.block_size:]
        logits = model(ctx)
        probs  = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())
```

Line 7 picks the character with the single highest score for greedy decoding. Lines 17 and 18 turn the scores into probabilities and draw a winner randomly for sampling.

![Code flow: model(ctx) gives logits, argmax or multinomial picks next_id, which is appended to context](https://jackluu.io/book/assets/diagrams/ch15-code-flow.png)

*Figure 15.3: The generation loop predicts one character at a time and appends it to the context.*

You may have noticed we are using `ids[:, -cfg.block_size:]`, which will grab only the last `block_size` tokens of our current text (the context) before we pass it to the model. We need to keep this as our model was trained to only understand a max of our block size which is 128. So we want to give our model no more than that. Notice that we use `logits[:, -1, :]` since we only care about what score the next letter will be at the last position.

As you can see there are two ways to go about this using `ROMEO:\n` as our prompt.

```text
$ python src/ch14_generate_greedy.py
--- GREEDY (always picks highest-score token) ---
ROMEO:
I will the come the some the stand the son,
And the so shall the see the shall the see the stand
...
--- SAMPLING (picks randomly from distribution) ---
ROMEO:
My must greather, but what it? whom, he ere.

EBRUTUS:
Morcy there bagainVain, I will ever may
To epated as great that the appral tankswain
...
--- Observations ---
Greedy tends to be more repetitive.
Sampling is more varied but can make unexpected choices.
```

```text
$ python diagrams/charts/ch15_probabilities.py
Saved ch15-probabilities.png and ch15-probabilities.pdf
```

![Bars comparing greedy and sampling on the same probabilities](https://jackluu.io/book/assets/diagrams/ch15-probabilities.png)

*Figure 15.4: Greedy always picks the tallest bar. Sampling can pick any bar, in proportion to its height.*

**What just happened:**

- The greedy approach quickly got stuck in a repetitive loop ("the shall the shall...").
- The sampling approach produced varied, non-repetitive text, but it included some strange spellings ("greather", "bagainVain") because it occasionally picked low-probability characters.

### Shape Check

Table 15.1 lists the shapes used during the character prediction step.

**Table 15.1:** Tensors used during the text generation loop.

| Tensor | Shape | What it means |
|--------|-------|---------------|
| `logits[:, -1, :]` | `[1, 65]` | 65 vocabulary scores for the final character in the sequence. |
| `next_id` | `[1, 1]` | 1 chosen character ID. |

## Try It

> **Try It**
>
> Different starting prompts create different context for the model. The script below shows how sampling handles new prompts.
>
> ```python title="src/examples/ch15_different_prompts.py (excerpt)" linenums="1" hl_lines="6 9"
> # ...
> encode_fn = lambda s: encode(s, char_to_id)
> decode_fn = lambda ids: decode(ids, id_to_char)
>
> print(f"--- Prompt: 'KING:\\n' ---")
> print(generate_sample(model, "KING:\n", encode_fn, decode_fn, cfg, 50))
>
> print(f"\n--- Prompt: 'JULIET:\\n' ---")
> print(generate_sample(model, "JULIET:\n", encode_fn, decode_fn, cfg, 50))
> ```
>
> Lines 6 and 9 generate new text starting from two different prompts.
>
> ```console title="Terminal"
> $ python src/examples/ch15_different_prompts.py
> --- Prompt: 'KING:\n' ---
> KING:
> My mine gore to kink on Villence,
> For my love grin
>
> --- Prompt: 'JULIET:\n' ---
> JULIET:
> What reason this bagainVain, I will evers,
> As do p
>
> ```

## Key Takeaways

- Greedy decoding always chooses the highest-scoring token, which leads to predictable but repetitive text.
- Sampling chooses the next token randomly based on the probability distribution, and the text comes out varied and natural.
- The generation loop works by predicting one character, appending it to the context, and repeating the process.
- We crop the input context to the model's maximum `block_size` before each prediction.

## Check Your Understanding

1. Why does greedy decoding often get stuck repeating the same phrase?
2. What PyTorch function do we use to randomly pick a token based on its probabilities?
3. Why do we slice the logits tensor with `[:, -1, :]` in the generation loop?

## Assignment

> **Assignment 15: Which half changes**
>
> **About 15 minutes. You change one number and write no new code.** Work in this chapter's Colab notebook or on your own laptop.
>
> 1. **Run it.** Run `src/ch14_generate_greedy.py` as it is. Copy the text under the GREEDY heading and the text under the SAMPLING heading.
> 2. **Change one number.** In the demo part of the file, find `torch.manual_seed(42)`. The seed fixes the random draws, so every run of the script picks the same characters. Change `42` to any whole number you like and run the script again.
> 3. **Compare.** Which of the two texts changed? Which one is the same, character for character? In the greedy text, which words keep coming back?
> 4. **Explain it.** In two or three sentences a manager could follow, say which rule you would choose for a task that must give the same answer every time, and what you give up by choosing it.
>
> **Hand in:** both pairs of texts, your answers to step 3, and your sentences.

## Further Reading

**Why always picking the likeliest word goes wrong.** Simply selecting the next token with the highest score at each time step, known as greedy decoding in Chapter 15, yields bland and repetitive output. The paper shows that this is the case because natural language texts are not composed of the locally most predictable word at each time step. They propose an alternative that samples from the smallest set of tokens with probabilities that sum to a given proportion. Choice of next token is just as important as training.


Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2019). *The curious case of neural text degeneration* (arXiv:1904.09751). arXiv. https://doi.org/10.48550/arXiv.1904.09751

---

### `src/ch14_generate_greedy.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch14_generate_greedy.py"   # a cell has none, and the file uses it to find the text

"""
Generate text using greedy decoding and sampling.
This file belongs to Chapter 15.
Run: python src/ch14_generate_greedy.py
"""
import os
import sys
import torch
import torch.nn.functional as F


from src.utils.config import GPTConfig
from src.ch09_gpt_model import GPT

# Settings
CHECKPOINT_PATH = "checkpoints/model.pt"
DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")

# --- The Idea ---

def generate_greedy(model, prompt, encode, decode, cfg, max_new_tokens=200):
    # Always picks the token with the highest predicted score
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    for _ in range(max_new_tokens):
        ctx     = ids[:, -cfg.block_size:]
        logits  = model(ctx)
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids     = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

def generate_sample(model, prompt, encode, decode, cfg, max_new_tokens=200):
    # Picks the next token randomly based on the model's probabilities
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    for _ in range(max_new_tokens):
        ctx    = ids[:, -cfg.block_size:]
        logits = model(ctx)
        probs  = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 15: Greedy Decoding vs Sampling\n")

    if not os.path.exists(DATA_PATH):
        print("ERROR: shakespeare.txt not found. Run download_data.py")
        sys.exit(1)

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()

    chars = sorted(set(text))
    char_to_id  = {ch: i for i, ch in enumerate(chars)}
    id_to_char  = {i: ch for i, ch in enumerate(chars)}

    encode = lambda s: [char_to_id[c] for c in s if c in char_to_id]
    decode = lambda ids: "".join([id_to_char[i] for i in ids])

    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location="cpu", weights_only=False
    )
    cfg = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    trained_steps = checkpoint["step"]
    val_loss      = checkpoint["val_loss"]
    print(f"\nModel loaded: trained for {trained_steps} steps, "
          f"val loss = {val_loss:.4f}")

    prompt = "ROMEO:\n"
    print(f"\nPrompt: {repr(prompt)}")
    print("Generating 200 characters each...\n")

    with torch.no_grad():
        greedy_text = generate_greedy(
            model, prompt, encode, decode, cfg, max_new_tokens=200
        )
        sample_text = generate_sample(
            model, prompt, encode, decode, cfg, max_new_tokens=200
        )

    print("--- GREEDY (always picks highest-score token) ---")
    print(greedy_text)

    print("\n--- SAMPLING (picks randomly from distribution) ---")
    print(sample_text)

    print("\n--- Observations ---")
    print("Greedy tends to be more repetitive.")
    print("Sampling is more varied but can make unexpected choices.")
    print("For better output, see Chapter 16 (temperature and top-k).")

---

### `src/examples/ch15_different_prompts.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch15_different_prompts.py"   # a cell has none, and the file uses it to find the text

"""Generate text from different starting prompts using sampling."""
import os
import sys
import torch

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

from src.ch03_tokenizer import build_vocab, encode, decode
from src.ch09_gpt_model import GPT
from src.ch14_generate_greedy import generate_sample

def main():
    torch.manual_seed(42)
    checkpoint_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "checkpoints", "model.pt"
    )
    
    if not os.path.exists(checkpoint_path):
        print("Checkpoint not found.")
        return
        
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    
    cfg = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    
    text_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "src", "data", "shakespeare.txt"
    )
    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read()
    chars, char_to_id, id_to_char = build_vocab(text)
    
    encode_fn = lambda s: encode(s, char_to_id)
    decode_fn = lambda ids: decode(ids, id_to_char)
    
    print(f"--- Prompt: 'KING:\\n' ---")
    print(generate_sample(model, "KING:\n", encode_fn, decode_fn, cfg, 50))
    
    print(f"\n--- Prompt: 'JULIET:\\n' ---")
    print(generate_sample(model, "JULIET:\n", encode_fn, decode_fn, cfg, 50))

if __name__ == "__main__":
    main()